In [ ]:
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from pathlib import Path

import csv
import hashlib
import math
import os
import random
import traceback

import matplotlib.pyplot as plt
import numpy as np

from main import load_graph_and_communities, run_configured_sicp_intermediates
from removal import (
    count_communities_in_hedges_removal,
    count_ic_hedges,
    degree_based_removal,
    filtered_hyperdegree_removal,
    hyperdegree_based_removal,
    random_based_removal,
    remove_nodes,
    responsibility_weighted_hdeg_removal,
 )

# -----------------------------
# Config
# -----------------------------
dataset = "Algebra"
tau = 2
p_values = [0.05, 0.10, 0.20]
beta = 0.02
T = 25
runs = 10
seed_iterations = 2000
replicates = 8
rng_seed = 175

# Excluded: avg_hyperedge_size, ic_hedges_to_hdeg
strategies = [
    ("count_communities_in_hedges", True),
    ("count_ic_hedges", True),
    ("responsibility_weighted_hdeg", True),
    ("filtered_hyperdegree", True),
    ("hyperdegree", False),
    ("degree", False),
    ("random", False),
]

out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
replicate_file = out_dir / "algebra_strategy_replicates.csv"
summary_file = out_dir / "algebra_strategy_summary.csv"
errors_file = out_dir / "algebra_strategy_errors.csv"

base_graph, base_communities = load_graph_and_communities(dataset, tau=tau, verbose=True)
N = len(set.union(*base_graph.values()))

def stable_seed(*parts):
    payload = "|".join(str(x) for x in parts).encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "little") & 0x7FFFFFFF

def nodes_for_strategy(strategy_name, graph, communities, K):
    if strategy_name == "count_communities_in_hedges":
        return count_communities_in_hedges_removal(graph, communities, K)
    if strategy_name == "count_ic_hedges":
        return count_ic_hedges(graph, communities, K)
    if strategy_name == "responsibility_weighted_hdeg":
        return responsibility_weighted_hdeg_removal(graph, communities, K)
    if strategy_name == "filtered_hyperdegree":
        return filtered_hyperdegree_removal(graph, communities, K)
    if strategy_name == "hyperdegree":
        return hyperdegree_based_removal(graph, K)
    if strategy_name == "degree":
        return degree_based_removal(graph, K)
    if strategy_name == "random":
        return random_based_removal(graph, K)
    raise ValueError(f"Unknown strategy: {strategy_name}")

def run_one_job(job):
    strategy_name, needs_communities, p, rep_idx = job
    try:
        graph = {eid: set(nodes) for eid, nodes in base_graph.items()}
        communities = dict(base_communities)

        K = int(p * float(N))
        random.seed(stable_seed("remove", dataset, strategy_name, p, rep_idx, rng_seed))

        if needs_communities:
            nodes_to_remove = nodes_for_strategy(strategy_name, graph, communities, K)
        else:
            nodes_to_remove = nodes_for_strategy(strategy_name, graph, None, K)

        graph_removed = remove_nodes(graph, nodes_to_remove)
        if not graph_removed:
            raise ValueError("remove_nodes produced empty graph")

        remaining_nodes = len(set.union(*graph_removed.values()))
        if remaining_nodes == 0:
            raise ValueError("zero remaining nodes")

        mean_counts = run_configured_sicp_intermediates(
            graph_removed,
            seed_iterations=seed_iterations,
            beta=beta,
            T=T,
            runs=runs,
            rng_seed=rng_seed + rep_idx * 1000,
        )

        if len(mean_counts) < T + 1:
            tail = mean_counts[-1] if mean_counts else 0.0
            mean_counts = mean_counts + [tail] * (T + 1 - len(mean_counts))
        elif len(mean_counts) > T + 1:
            mean_counts = mean_counts[: T + 1]

        final_count = float(mean_counts[-1])
        final_pct = 100.0 * final_count / remaining_nodes

        return {
            "ok": True,
            "dataset": dataset,
            "strategy": strategy_name,
            "p": p,
            "replicate": rep_idx,
            "K_removed": K,
            "remaining_nodes": remaining_nodes,
            "final_mean_infected_count": final_count,
            "final_mean_infected_pct": final_pct,
        }
    except Exception as exc:
        return {
            "ok": False,
            "dataset": dataset,
            "strategy": strategy_name,
            "p": p,
            "replicate": rep_idx,
            "error": f"{type(exc).__name__}: {exc}",
            "traceback": traceback.format_exc(limit=1).strip(),
        }

jobs = [
    (strategy_name, needs_communities, p, rep_idx)
    for p in p_values
    for strategy_name, needs_communities in strategies
    for rep_idx in range(1, replicates + 1)
]

rep_rows = []
err_rows = []
max_workers = max(1, min(len(jobs), os.cpu_count() or 4))
print(f"Running {len(jobs)} jobs with up to {max_workers} workers")

used_engine = "process"
try:
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one_job, job) for job in jobs]
        for i, fut in enumerate(as_completed(futures), start=1):
            res = fut.result()
            if res["ok"]:
                rep_rows.append(res)
                print(f"[{i}/{len(jobs)}] ok  | {res['strategy']:<30} p={res['p']:.2f} rep={res['replicate']}")
            else:
                err_rows.append(res)
                print(f"[{i}/{len(jobs)}] fail| {res['strategy']:<30} p={res['p']:.2f} rep={res['replicate']}")
except Exception as exc:
    used_engine = "thread"
    print(f"Process pool failed ({exc}). Falling back to threads.")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(run_one_job, job) for job in jobs]
        for i, fut in enumerate(as_completed(futures), start=1):
            res = fut.result()
            if res["ok"]:
                rep_rows.append(res)
                print(f"[{i}/{len(jobs)}] ok  | {res['strategy']:<30} p={res['p']:.2f} rep={res['replicate']}")
            else:
                err_rows.append(res)
                print(f"[{i}/{len(jobs)}] fail| {res['strategy']:<30} p={res['p']:.2f} rep={res['replicate']}")

if not rep_rows:
    raise RuntimeError("No successful jobs completed.")

rep_fields = [
    "dataset",
    "strategy",
    "p",
    "replicate",
    "K_removed",
    "remaining_nodes",
    "final_mean_infected_count",
    "final_mean_infected_pct",
]
with replicate_file.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=rep_fields)
    w.writeheader()
    w.writerows(rep_rows)

grouped = defaultdict(list)
meta = {}
for r in rep_rows:
    key = (r["strategy"], r["p"])
    grouped[key].append(float(r["final_mean_infected_pct"]))
    meta[key] = (r["K_removed"], r["remaining_nodes"])

summary_rows = []
for (strategy_name, p), vals in grouped.items():
    n = len(vals)
    mean_v = float(np.mean(vals))
    std_v = float(np.std(vals, ddof=1)) if n > 1 else 0.0
    sem_v = std_v / math.sqrt(n) if n > 1 else 0.0
    k_removed, remaining_nodes = meta[(strategy_name, p)]
    summary_rows.append({
        "dataset": dataset,
        "strategy": strategy_name,
        "p": p,
        "K_removed": k_removed,
        "remaining_nodes": remaining_nodes,
        "n_replicates": n,
        "final_mean_infected_pct": mean_v,
        "final_std_infected_pct": std_v,
        "final_sem_infected_pct": sem_v,
    })

summary_rows = sorted(summary_rows, key=lambda r: (r["p"], r["strategy"]))
sum_fields = [
    "dataset",
    "strategy",
    "p",
    "K_removed",
    "remaining_nodes",
    "n_replicates",
    "final_mean_infected_pct",
    "final_std_infected_pct",
    "final_sem_infected_pct",
]
with summary_file.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=sum_fields)
    w.writeheader()
    w.writerows(summary_rows)

if err_rows:
    err_fields = ["dataset", "strategy", "p", "replicate", "error", "traceback"]
    with errors_file.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=err_fields)
        w.writeheader()
        w.writerows(err_rows)

p_unique = sorted({r["p"] for r in summary_rows})
s_unique = sorted({r["strategy"] for r in summary_rows})
x = np.arange(len(s_unique))
width = 0.8 / len(p_unique)

fig, ax = plt.subplots(figsize=(max(12, 1.2 * len(s_unique) + 4), 6))
for i, p in enumerate(p_unique):
    means = []
    errs = []
    for s in s_unique:
        hit = next((r for r in summary_rows if r["strategy"] == s and r["p"] == p), None)
        if hit is None:
            means.append(np.nan)
            errs.append(0.0)
        else:
            means.append(hit["final_mean_infected_pct"])
            errs.append(hit["final_sem_infected_pct"])

    offset = (i - (len(p_unique) - 1) / 2.0) * width
    ax.bar(
        x + offset,
        means,
        width=width,
        yerr=errs,
        capsize=4,
        alpha=0.9,
        label=f"p={int(100 * p)}%",
    )

ax.set_xticks(x)
ax.set_xticklabels(s_unique, rotation=30, ha="right")
ax.set_ylabel("Final infected prevalence (%)")
ax.set_title(f"Algebra: final prevalence by strategy ({used_engine} parallel, SEM error bars)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(title="Removal fraction")
plt.tight_layout()
plt.show()

print(f"Replicates saved: {replicate_file} ({len(rep_rows)} rows)")
print(f"Summary saved   : {summary_file} ({len(summary_rows)} rows)")
if err_rows:
    print(f"Errors saved    : {errors_file} ({len(err_rows)} rows)")

Loaded dataset 'Algebra' (edge-labeled) with 1268 hyperedges and 10 communities
  communities sizes: min=2, max=88, avg=42.300
Running 168 jobs with up to 20 workers
Process pool failed (A process in the process pool was terminated abruptly while the future was running or pending.). Falling back to threads.
